Part 3: Solve a problem together [30 minutes]
Q3: Small Expressions
Imagine you can call only the following three functions:

f(x): Adds one to an integer x. For example, f(5) is 6.
g(x): Doubles an integer x and then subtracts one. For example, g(5) is 9.
h(x, y): Concatenates the digits of two different positive integers x and y. For example, h(789, 12) evaluates to 78912 and h(12, 789) evaluates to 12789.
Definition: A small expression is a call expression that contains only f, g, h, the number 5, and parentheses. All of these can be repeated. For example, h(f(g(5)), f(f(5))) is a small expression that evaluates to 107.

What's the shortest small expression you can find that evaluates to 2026?

In [ ]:
"""
Q3: Small Expressions
======================
只能用三个函数构造表达式，求最短得到 2026 的 small expression：
    f(x) = x + 1
    g(x) = 2*x - 1
    h(x, y) = 拼接 x、y 的数字（int(str(x)+str(y))）
基数只有 5。

实现思路：
    把每个可达整数看成图中的一个节点，用 Dijkstra 按"函数调用次数"求最短。
    - f / g 是一元边，代价 +1
    - h 是二元边：把当前节点与所有已 settle 的节点两两拼接，代价 = 两边调用数之和 + 1
    由于所有边权为正，Dijkstra 保证按调用次数全局最优。
"""

import heapq


def f(x):
    return x + 1


def g(x):
    return 2 * x - 1


def h(x, y):
    return int(str(x) + str(y))


def shortest_expr(target, cap=99999, trace=False):
    """返回 (调用次数, 表达式字符串)。cap 限制中间值位数，避免搜索空间爆炸。
    因为目标是 4 位数，用 h 拼出它所需的各个片段都 <= cap。
    trace=True 时打印 Dijkstra 的探索过程（每弹出一个被确定的节点就输出一行）。"""
    best = {5: (0, "5")}            # value -> (calls, expr) 当前已知最优
    settled = {}                   # value -> (calls, expr) 已确定最优
    pq = [(0, 5, "5")]             # (calls, value, expr)

    def relax(nv, nc, ne):
        if 1 <= nv <= cap and nv not in settled:
            if nv not in best or best[nv][0] > nc:
                best[nv] = (nc, ne)
                heapq.heappush(pq, (nc, nv, ne))

    popped = 0
    while pq:
        calls, val, expr = heapq.heappop(pq)
        if val in settled:
            continue
        settled[val] = (calls, expr)
        popped += 1
        if trace:
            print(f"[确定 #{popped:>4d}] calls={calls:<2d} value={val:<7d} expr={expr}")
        if val == target:
            if trace:
                print(f"命中目标 {target}，共确定 {popped} 个节点后停止。")
            return settled[val]

        # 一元边
        relax(f(val), calls + 1, "f(" + expr + ")")
        relax(g(val), calls + 1, "g(" + expr + ")")

        # 二元边 h：与所有已 settle 的节点两两拼接（x、y 需为不同的两个数）
        sval = str(val)
        for ov, (oc, oe) in settled.items():
            if ov == val:
                continue
            relax(h(val, ov), calls + oc + 1, "h(" + expr + "," + oe + ")")
            relax(h(ov, val), calls + oc + 1, "h(" + oe + "," + expr + ")")

    return None


def verify(expr):
    """用真实函数定义求值，确认表达式结果。"""
    return eval(expr, {"f": f, "g": g, "h": h})


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="求最短 small expression")
    parser.add_argument("target", nargs="?", type=int, default=2026,
                        help="目标整数（默认 2026）")
    parser.add_argument("--trace", action="store_true", default=False,
                        help="打印搜索探索过程（默认关闭）")
    # 用 parse_known_args 忽略 Jupyter/IPython kernel 注入的 --f=...json 等额外参数
    args, _ = parser.parse_known_args()

    calls, expr = shortest_expr(args.target, trace=args.trace)
    if args.trace:
        print("-" * 60)
    print(f"目标      : {args.target}")
    print(f"最少调用数: {calls}")
    print(f"表达式    : {expr}")
    print(f"求值验证  : {verify(expr)}  (匹配: {verify(expr) == args.target})")
    print(f"字符串长度: {len(expr)}")


usage: ipykernel_launcher.py [-h] [--trace] [target]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/junyiwu/Library/Jupyter/runtime/kernel-v3674ac899e5f9b7ff9569721dc9e652238a9eded6.json


SystemExit: 2

/Users/junyiwu/miniconda3/envs/torch/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
